# 🩺 Příprava dat pro klasifikační modely (Diabetes Dataset)
**Kurz: Data Science / Machine Learning**  
**Sekce: Homework mezi dny 2 a 3 (Klasifikace)**

### Cíl cvičení:
Vytvořit robustní pipeline pro přípravu a předzpracování medicínských dat (`diabetes.csv`), odhalit skryté chybějící hodnoty (biologicky nemožné nuly), zvolit správnou strategii imputace na základě rozdělení pravděpodobnosti a data standardizovat pro klasifikační algoritmy.

---
### Přehled proměnných v datasetu:
- `pregnancies`: Počet předchozích těhotenství
- `glucose`: Hladina glukózy 2 hodiny po zátěžovém orálním testu (oGTT) v mg/dl
- `blood_pressure`: Diastolický krevní tlak v mm Hg
- `skin_thickness`: Tloušťka kožní řasy na tricepsu v mm
- `insulin`: Hladina inzulinu 2 hodiny po orálním testu v mIU/ml
- `bmi`: Body Mass Index (index tělesné hmotnosti) v kg/m²
- `diabetes_pedigree_function`: Skóre genetické predispozice k diabetu na základě rodinné anamnézy
- `age`: Věk pacientky v letech
- `outcome`: Diagnóza (1 = diabetes, 0 = bez diabetu)


In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Nastavení stylu grafů
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Krok 1 & 2: Načtení dat a normalizace názvů sloupců
Převedeme názvy sloupců do formátu `snake_case` (malá písmena, slova oddělená podtržítkem).

In [ ]:
# Načtení datové sady
df = pd.read_csv('data/diabetes.csv')

# Normalizace názvů sloupců
col_mapping = {
    'Pregnancies': 'pregnancies',
    'Glucose': 'glucose',
    'BloodPressure': 'blood_pressure',
    'SkinThickness': 'skin_thickness',
    'Insulin': 'insulin',
    'BMI': 'bmi',
    'DiabetesPedigreeFunction': 'diabetes_pedigree_function',
    'Age': 'age',
    'Outcome': 'outcome'
}
df.rename(columns=col_mapping, inplace=True)
df.columns = [c.strip().lower() for c in df.columns]

print(f"Rozměry datasetu: {df.shape[0]} řádků, {df.shape[1]} sloupců")
df.head()


## Krok 3 & 4: Kontrola NaN a distribuce cílové třídy `outcome`

In [ ]:
# Kontrola explicitních chybějících hodnot
print("Explicitní NaN hodnoty v datasetu:")
print(df.isna().sum())

# Četnost cílové proměnné
counts = df['outcome'].value_counts()
ratios = df['outcome'].value_counts(normalize=True)

print("\nDistribuce třídy Outcome:")
for cls in [0, 1]:
    lbl = 'Bez diabetu (0)' if cls == 0 else 'Diabetes (1)'
    print(f"{lbl}: {counts[cls]} pacientek ({ratios[cls]*100:.1f} %)")


## Krok 5: Analýza rozdělení a biologicky nemožné nuly
V medicínských datech často chybějící údaje zapisovali jako `0`.
Žijící pacient nemůže mít nulovou glykémii, krevní tlak, tloušťku kožní řasy, inzulin po testu ani BMI.

In [ ]:
zero_cols = ['glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi']

print("Počet nulových hodnot v medicínských proměnných:")
for col in zero_cols:
    zc = (df[col] == 0).sum()
    print(f"{col:20s}: {zc:3d} nul ({zc / len(df) * 100:.2f} %)")


## Krok 6 & 7: Nahrazení nul za NaN a imputace dle rozdělení
Nahradíme nuly za `np.nan`, spočítáme šikmost (skewness) a zvolíme:
- **Průměr (mean)** pro přibližně normální rozdělení (|skew| < 0.5)
- **Medián (median)** pro šikmá rozdělení (|skew| >= 0.5)

In [ ]:
df_imputed = df.copy()

# Nahrazení nul za NaN
for col in zero_cols:
    df_imputed[col] = df_imputed[col].replace(0, np.nan)

print("Vzniklé chybějící hodnoty (NaN):")
print(df_imputed[zero_cols].isna().sum())

print("\n--- Strategie imputace dle šikmosti (skewness) ---")
for col in zero_cols:
    val_series = df_imputed[col].dropna()
    sk = val_series.skew()
    if abs(sk) < 0.5:
        method = 'mean (průměr)'
        fill_val = val_series.mean()
    else:
        method = 'median (medián)'
        fill_val = val_series.median()
    df_imputed[col] = df_imputed[col].fillna(fill_val)
    print(f"{col:20s} | skew: {sk:+.3f} | metoda: {method:18s} | hodnota: {fill_val:.2f}")

print(f"\nCelkový zbývající počet NaN v datech: {df_imputed.isna().sum().sum()}")


## Krok 8: Korelační analýza

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_imputed.corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='Blues', square=True, cbar_kws={'shrink': .8})
plt.title('Korelační matice po imputaci', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

print("Korelace jednotlivých příznaků s výskytem diabetu (outcome):")
print(corr['outcome'].sort_values(ascending=False))


## Krok 9 & 10: Škálování (Standardizace) a uložení datasetu
Použijeme `StandardScaler` (z-score normalizace: průměr=0, směrodatná odchylka=1). Cílovou proměnnou `outcome` ponecháme binární (0 / 1).

In [ ]:
features = [c for c in df_imputed.columns if c != 'outcome']
scaler = StandardScaler()
scaled_vals = scaler.fit_transform(df_imputed[features])

df_scaled = pd.DataFrame(scaled_vals, columns=features)
df_scaled['outcome'] = df_imputed['outcome'].values

# Uložení do souboru diabetes_scaled.csv
df_scaled.to_csv('data/diabetes_scaled.csv', index=False)
print("Uloženo: data/diabetes_scaled.csv")
df_scaled.head()
